In [ ]:
from pathlib import Path

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
import sys
import json
from shapely.geometry import shape
from hotelling.spatial.admin import join_lor_names

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from hotelling.spatial.boundaries import load_boundary

PATH_RAW = REPO_ROOT / Path('data/raw')
PATH_PROCESSED = REPO_ROOT / Path('data/processed')

# Midpoint table (center coordinates)
zensus = gpd.read_parquet(PATH_RAW / 'zensus2022_grid.parquet')
zensus_filtered = gpd.read_parquet(PATH_RAW / 'zensus2022_grid_filtered.parquet')
lor = gpd.read_parquet(PATH_PROCESSED / 'lor.parquet')

# CRITICAL FIX: berlin.geojson has EPSG:3035 coordinates but geopandas auto-detects as EPSG:4326
# We must force the correct CRS instead of transforming from the wrong one
with open(PATH_RAW / 'city_boundary_Berlin.geojson', 'r') as f:
    berlin_json = json.load(f)
berlin = gpd.GeoDataFrame([1], geometry=[shape(berlin_json['geometry'])], crs='EPSG:3035')

boundary = load_boundary(PATH_RAW / 'relation_boundary_14983.geojson')

In [ ]:
# Load pop_grid

grid = gpd.read_parquet(PATH_PROCESSED / 'pop_grid.parquet')

# Build squares from points of grid
grid['geometry'] = grid.apply(lambda row: row.geometry.buffer(50, cap_style='square'), axis=1)
grid['index'] = grid.index

In [ ]:
IHK = pd.read_csv(PATH_RAW / '2023_12_IHK_Berlin_Gewerbedaten.csv')

# To geodataframe based on latitude and longitude columns
IHK_gdf = gpd.GeoDataFrame(
    IHK,
    geometry=gpd.points_from_xy(IHK['longitude'], IHK['latitude']),
    crs='EPSG:4326'
).to_crs('EPSG:3035')

In [ ]:
# from e.g. '1 - 3 Beschäftigte' to mean(1,3) = 2
def parse_employees_range(range_str):
    if pd.isna(range_str):
        return np.nan
    
    if range_str == 'unbekannt':
        return np.nan
    
    try:
        parts = range_str.split('-')
        if len(parts) == 2:
            low = int(parts[0].strip())
            high = int(parts[1].strip().split()[0])  # Remove 'Beschäftigte'
            return (low + high) / 2
        else:
            return int(range_str.strip().split()[0])  # Single value case
    except Exception as e:
        print(f"Error parsing '{range_str}': {e}")
        return np.nan

IHK_gdf['empl'] = IHK_gdf['employees_range'].apply(parse_employees_range)

In [ ]:
# Assign to grid cells - preserving grid cell ID in IHK_gdf for filtering
IHK_grid = gpd.sjoin(IHK_gdf, grid, how='left', predicate='within')

# Aggregate employment by grid cell
empl_by_cell = IHK_grid.groupby('index')['empl'].sum().reset_index()
empl_by_cell.columns = ['index', 'empl']

# Merge employment data back to grid
grid = grid.reset_index(drop=True).merge(empl_by_cell, left_index=True, right_on='index', how='left')
grid['empl'] = grid['empl'].fillna(0)

In [ ]:
# Load the gebaeude and stadtstruktur data
gebaeude = gpd.read_file(PATH_RAW / 'gebaeude.gpkg')
stadtstruktur = gpd.read_file(PATH_RAW / 'stadtstruktur.gpkg')
zentren_fma = gpd.read_file(PATH_RAW / 'zentren.gpkg', layer="zentren_fma")
zentren_zh = gpd.read_file(PATH_RAW / 'zentren.gpkg', layer="zentren_zh")

In [ ]:
# Step 1: within sjoin
filtered_gebaeude = gebaeude[gebaeude['bezeich'] == 'AX_Gebaeude'].copy()

sjoin_within = gpd.sjoin(
    filtered_gebaeude,
    stadtstruktur,
    how='left',
    predicate='within'
)

# Get building IDs with no match from within
within_match_counts = sjoin_within.groupby(level=0)['index_right'].apply(lambda x: x.notna().sum())
no_match_within = within_match_counts[within_match_counts == 0].index.tolist()

# Step 2: for unmatched buildings, try intersects sjoin
unmatched_gebaeude = filtered_gebaeude.loc[no_match_within].copy()
sjoin_intersects = gpd.sjoin(
    unmatched_gebaeude,
    stadtstruktur,
    how='left',
    predicate='intersects'
)

# Step 3: combine results - keep only first match from within, then fill with first match from intersects
# Map stadtstruktur ID for each building
stadtstruktur_match = {}

for building_id in filtered_gebaeude.index:
    # Try within first
    if building_id in sjoin_within.index:
        within_row = sjoin_within.loc[[building_id]].iloc[0]
        if pd.notna(within_row['index_right']):
            stadtstruktur_match[building_id] = (int(within_row['index_right']), 'within')
            continue
    
    # If no within match, try intersects
    if building_id in sjoin_intersects.index:
        intersects_row = sjoin_intersects.loc[[building_id]].iloc[0]
        if pd.notna(intersects_row['index_right']):
            stadtstruktur_match[building_id] = (int(intersects_row['index_right']), 'intersects')
            continue
    
    # If still no match
    stadtstruktur_match[building_id] = (np.nan, 'no_match')

# Create result by merging all columns from both datasets
gebaeude_with_match = filtered_gebaeude.copy()
gebaeude_with_match['stadtstruktur_id'] = gebaeude_with_match.index.map(lambda x: stadtstruktur_match[x][0])
gebaeude_with_match['match_type'] = gebaeude_with_match.index.map(lambda x: stadtstruktur_match[x][1])

# Extract building geometry before merge to avoid suffix issues
building_geoms = gebaeude_with_match[['geometry']].copy()
gebaeude_with_match_no_geom = gebaeude_with_match.drop(columns=['geometry'])

# Merge with stadtstruktur to get all columns (no geometry conflict)
stadtstruktur_no_geom = stadtstruktur.reset_index().rename(columns={'index': 'index_right'}).drop(columns=['geometry'])
gebaeude_stadtstruktur = gebaeude_with_match_no_geom.merge(
    stadtstruktur_no_geom,
    left_on='stadtstruktur_id',
    right_on='index_right',
    how='left'
)

# Add building geometry back
gebaeude_stadtstruktur['geometry'] = building_geoms['geometry']
gebaeude_stadtstruktur = gpd.GeoDataFrame(gebaeude_stadtstruktur, geometry='geometry', crs=filtered_gebaeude.crs)

no_stadtstruktur = gebaeude_stadtstruktur[gebaeude_stadtstruktur['stadtstruktur_id'].isna()].index.tolist()

print(f"Within matches: {len(gebaeude_stadtstruktur[gebaeude_stadtstruktur['match_type'] == 'within'])}")
print(f"Intersects matches: {len(gebaeude_stadtstruktur[gebaeude_stadtstruktur['match_type'] == 'intersects'])}")
print(f"No matches: {len(no_stadtstruktur)}")


In [ ]:
df_to_plot = gebaeude_stadtstruktur[gebaeude_stadtstruktur['aog'].isna()]

# Plot with berlin boundary
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(20, 20))
berlin.to_crs('EPSG:25833').plot(ax=ax, color='none', edgecolor='black', linewidth=1)
df_to_plot.plot(ax=ax, color='red', markersize=10)
plt.title('Buildings with no stadtstruktur match (red) and Berlin boundary')
plt.show()


In [ ]:
# Unique count of bezgfk values for buildings with no stadtstruktur match
df_to_plot['bezgfk'].value_counts()

# If bezgfk == "Tiefgarage" -> aog = 0, bezgfk == "Garage" -> aog = 0, "Gebäude zum Parken" -> aog = 0, "Umformer" -> aog = 1, "Schutzbunker" -> aog = 0
# "Heizwerk" -> aog = 1, "Gebäude zur Elektrizitätsversorgung" -> aog = 1, "Parkdeck" -> aog = 0, "Speichergebäude" -> aog = 1, "Gebäude für Vorratshaltung" -> aog = 1
# "Pumpstation" -> aog = 0, "Lagerhalle, Lagerschuppen, Lagerhaus" -> aog = 1, "Gebäude zum Sportplatz" -> aog = 1, "Sport-, Turnhalle" -> aog = 1, "Gebäude für Sportzwecke" -> aog = 1,
# Wasserbehälter, Pumpwerk (nicht für Wasserversorgung) -> aog = 0
# Gebäude zur Wasserversorgung, Gebäude zur Abwasserbeseitigung, Hallenbad, Schuppen, Gartenhaus, Wasserwerk, Gebäude zur Abfallbehandlung, Gebäude für Land- und Forstwirtschaft, Bootshaus, Tierschauhaus, Müllbunker, Parkhaus -> aog = 1

# Create a mapping of bezgfk to aog based on the above logic
bezgfk_to_aog = {
    "Tiefgarage": 0,
    "Garage": 0,
    "Gebäude zum Parken": 0,
    "Umformer": 1,
    "Schutzbunker": 0,
    "Heizwerk": 1,
    "Gebäude zur Elektrizitätsversorgung": 1,
    "Parkdeck": 0,
    "Speichergebäude": 1,
    "Gebäude für Vorratshaltung": 1,
    "Pumpstation": 0,
    "Lagerhalle, Lagerschuppen, Lagerhaus": 1,
    "Gebäude zum Sportplatz": 1,
    "Sport-, Turnhalle": 1,
    "Gebäude für Sportzwecke": 1,
    "Wasserbehälter": 0,
    "Pumpwerk (nicht für Wasserversorgung)": 0,
    "Gebäude zur Wasserversorgung": 1,
    "Gebäude zur Abwasserbeseitigung": 1,
    "Hallenbad": 1,
    "Schuppen": 1,
    "Gartenhaus": 1,
    "Wasserwerk": 1,
    "Gebäude zur Abfallbehandlung": 1,
    "Gebäude für Land- und Forstwirtschaft": 1,
    "Bootshaus": 1,
    "Tierschauhaus": 1,
    "Müllbunker": 1,
    "Parkhaus": 1
}

# Apply the mapping to fill aog values for buildings with no stadtstruktur match
gebaeude_stadtstruktur['aog'] = gebaeude_stadtstruktur.apply(
    lambda row: bezgfk_to_aog.get(row['bezgfk'], np.nan) if pd.isna(row['aog']) else row['aog'],
    axis=1
)

In [ ]:
df_to_plot = gebaeude_stadtstruktur[gebaeude_stadtstruktur['aog'].isna()]

# Plot with berlin boundary
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(20, 20))
berlin.to_crs('EPSG:25833').plot(ax=ax, color='none', edgecolor='black', linewidth=1)
df_to_plot.plot(ax=ax, color='red')
plt.title('Buildings with no stadtstruktur match (red) and Berlin boundary')
plt.show()


In [ ]:
gebaeude_stadtstruktur[gebaeude_stadtstruktur['aog'].isna()]['bezgfk'].value_counts()

# -> Just remove these from the gebaude_stadtstruktur

gebaeude_stadtstruktur = gebaeude_stadtstruktur[~gebaeude_stadtstruktur['aog'].isna()]

# HOH to bool
gebaeude_stadtstruktur['hoh'] = gebaeude_stadtstruktur['hoh'].apply(lambda x: False if x == 'false' or x == '' else True)

In [ ]:
from hotelling.spatial.gebaeude_capacity import (
    get_efficiency_factor,
    compute_usable_floor_area,
    compute_employee_hard_cap,
)

gebaeude_stadtstruktur["efficiency"] = gebaeude_stadtstruktur.apply(
    lambda r: get_efficiency_factor(r["gfk"], bool(r["hoh"])), axis=1
)
gebaeude_stadtstruktur["usable_area_m2"] = gebaeude_stadtstruktur.apply(
    lambda r: compute_usable_floor_area(
        r['shape_area'],
        r["aog"],
        r["gfk"],
        bool(r["hoh"]),
    ), axis=1
)

gebaeude_stadtstruktur["employee_hard_cap"] = gebaeude_stadtstruktur.apply(
    lambda r: compute_employee_hard_cap(
        r["shape_area"],
        r["aog"],
        r["gfk"],
        bool(r["hoh"]),
    ), axis=1
)

gebaeude_stadtstruktur["effective_employees"] = gebaeude_stadtstruktur["employee_hard_cap"].apply(lambda x: np.floor(x) if x != np.inf else 0) 

In [ ]:
# take unique pairs of long and lat and count the employees for each pair on IHK_gdf

IHK_gdf['lon_lat'] = IHK_gdf.apply(lambda row: (row['longitude'], row['latitude']), axis=1)
empl_by_location = IHK_gdf.groupby('lon_lat')['empl'].sum().reset_index()
empl_by_location.columns = ['lon_lat', 'empl']

empl_by_location['count'] = IHK_gdf.groupby('lon_lat')['empl'].count().values

display(empl_by_location.sort_values('empl', ascending=False).head(25))

In [ ]:
IHK_gdf[IHK_gdf['lon_lat'] == (13.413762562, 52.520831532)]